# Module 9: exercises  (solutions)

All exercises use `students.csv` (predicting `exam_score`), except Exercise 3 which uses clean synthetic data.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Slope and intercept by formula.** Using $b_1=r\,s_y/s_x$ and $b_0=\bar y-b_1\bar x$, compute `b1` and `b0` for predicting `exam_score` from `study_hours_week`.

In [ ]:
import math

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression

students = pd.read_csv(DATA / "students.csv")
x = students["study_hours_week"]
y = students["exam_score"]

b1 = x.corr(y) * y.std() / x.std()
b0 = y.mean() - b1 * x.mean()

In [ ]:
slope, intercept = np.polyfit(x, y, 1)
assert abs(b1 - slope) < 1e-6 and abs(b0 - intercept) < 1e-6, f"got b1={b1}, b0={b0}"
print(f"✅ Correct! Each extra study hour goes with {b1:.2f} more points on average.")

## Exercise 2

**Measuring a line.** Write `mse(y, y_hat)` returning the mean squared error. Then compute `mse_guess` for the guess line $\hat y=40+2x$ and `mse_best` for the least-squares line from Exercise 1.
Which is smaller? (It must be `mse_best`; least squares minimises exactly this quantity.)

In [ ]:
def mse(y, y_hat):
    return float(np.mean((np.asarray(y) - np.asarray(y_hat)) ** 2))

mse_guess = mse(y, 40 + 2 * x)
mse_best = mse(y, b0 + b1 * x)

In [ ]:
assert mse([1, 2, 3], [1, 2, 3]) == 0 and mse([0, 0], [3, 4]) == 12.5
assert mse_best < mse_guess
assert abs(mse_best - 53.7) < 1.0, f"mse_best = {mse_best}"
print(f"✅ Correct! guess MSE {mse_guess:.1f} vs best-fit MSE {mse_best:.1f}")

## Exercise 3

**Gradient descent.** Complete `gradient_descent(x, y, lr, n_iter)`: start with `b0 = b1 = 0` and repeat `n_iter` times: compute the residuals, the two gradients
$-2\,\overline{e}$ and $-2\,\overline{e\,x}$, then update both parameters. Return `(b0, b1)`. Test it on the noise-free line $y=3+2x$.

In [ ]:
def gradient_descent(x, y, lr=0.1, n_iter=500):
    b0, b1 = 0.0, 0.0
    for _ in range(n_iter):
        resid = y - (b0 + b1 * x)
        grad_b0 = -2 * np.mean(resid)
        grad_b1 = -2 * np.mean(resid * x)
        b0 -= lr * grad_b0
        b1 -= lr * grad_b1
    return b0, b1

xs = np.linspace(-1, 1, 50)
ys = 3 + 2 * xs
fit0, fit1 = gradient_descent(xs, ys)

In [ ]:
assert abs(fit0 - 3) < 0.01 and abs(fit1 - 2) < 0.01, f"got ({fit0:.3f}, {fit1:.3f}); expected (3, 2)"
print(f"✅ Correct! Recovered intercept {fit0:.3f} and slope {fit1:.3f}")

## Exercise 4

**Multiple regression with statsmodels.** Fit `exam_score ~ study_hours_week + sleep_hours + attendance_pct + part_time_job` and store the fitted result as `model`.
Then store `r2` (its $R^2$), `job_coef` (the coefficient named `part_time_job[T.yes]`) and `job_ci` (a `(low, high)` tuple of its 95% interval).

In [ ]:
model = smf.ols("exam_score ~ study_hours_week + sleep_hours + attendance_pct + part_time_job", data=students).fit()
r2 = model.rsquared
job_coef = model.params["part_time_job[T.yes]"]
low, high = model.conf_int().loc["part_time_job[T.yes]"]
job_ci = (low, high)

In [ ]:
assert abs(r2 - 0.6916) < 0.001, f"R² = {r2}"
assert abs(job_coef - (-3.18)) < 0.01, f"job coefficient = {job_coef}"
assert job_ci[0] < job_coef < job_ci[1] < 0, "After adjusting, the interval for the job effect is entirely negative"
print(f"✅ Correct! Job coefficient {job_coef:.2f}, 95% CI ({job_ci[0]:.2f}, {job_ci[1]:.2f})")

## Exercise 5

**Normal equations.** Predict `exam_score` from `study_hours_week` and `sleep_hours`. Build `X` (a column of ones, then the two predictors) and solve $X^\top X\,\beta=X^\top y$ with `np.linalg.solve` to get `beta`
(a length-3 array: intercept, study, sleep).

In [ ]:
X = np.column_stack([np.ones(len(students)), students["study_hours_week"], students["sleep_hours"]])
beta = np.linalg.solve(X.T @ X, X.T @ y.to_numpy())

In [ ]:
assert X.shape == (200, 3)
ref = np.linalg.lstsq(X, y.to_numpy(), rcond=None)[0]
assert np.allclose(beta, ref, atol=1e-6), f"beta = {beta}"
print("✅ Correct!", beta.round(3))

## Exercise 6

**Predict a new student.** Using `model` from Exercise 4, predict the exam score for a student who studies 12 h/week, sleeps 7.5 h, attends 90% of classes and **has** a part-time job.
Store it in `pred`. (Hint: `model.predict(pd.DataFrame({...}))`; give the same column names the model was fitted with.)

In [ ]:
new = pd.DataFrame({"study_hours_week": [12], "sleep_hours": [7.5], "attendance_pct": [90], "part_time_job": ["yes"]})
pred = float(model.predict(new).iloc[0])

In [ ]:
manual = (model.params["Intercept"] + model.params["study_hours_week"] * 12 + model.params["sleep_hours"] * 7.5
          + model.params["attendance_pct"] * 90 + model.params["part_time_job[T.yes]"])
assert abs(pred - manual) < 1e-6, f"pred = {pred}"
print(f"✅ Correct! Predicted score: {pred:.1f}")

## Exercise 7

**Residual diagnostics.** From `model`: compute `rmse` (the square root of the mean of squared residuals) and `n_large`, the number of students whose absolute residual exceeds **2 × rmse**.
Roughly what share should that be if residuals are bell-shaped? (About 5%.)

In [ ]:
resid = model.resid
rmse = float(np.sqrt(np.mean(resid ** 2)))
n_large = int((resid.abs() > 2 * rmse).sum())

In [ ]:
assert abs(rmse - 5.82) < 0.02, f"rmse = {rmse}"
assert n_large == 10, f"got {n_large}"
print(f"✅ Correct! {n_large} of 200 = {n_large / 200:.1%} of residuals are beyond 2 RMSE (about 5% expected)")

## Exercise 8

**R² never falls.** The code below adds 10 columns of pure random noise. Fit the model `exam_score ~ study_hours_week + sleep_hours + attendance_pct` with and without the noise columns
and store each model's R² as `r2_base` and `r2_padded`, and each adjusted R² as `adj_base` and `adj_padded`.

In [ ]:
rng = np.random.default_rng(0)
noise = pd.DataFrame(rng.normal(size=(len(students), 10)), columns=[f"noise{i}" for i in range(10)])
with_noise = pd.concat([students, noise], axis=1)
base_formula = "exam_score ~ study_hours_week + sleep_hours + attendance_pct"

base = smf.ols(base_formula, students).fit()
padded = smf.ols(base_formula + " + " + " + ".join(noise.columns), with_noise).fit()
r2_base, r2_padded = base.rsquared, padded.rsquared
adj_base, adj_padded = base.rsquared_adj, padded.rsquared_adj

In [ ]:
assert r2_padded > r2_base, "R² cannot decrease when predictors are added"
assert abs(r2_base - 0.671) < 0.002 and abs(r2_padded - 0.7007) < 0.002
assert adj_padded < r2_padded and adj_base < r2_base, "adjusted R² is always below R²"
print(f"✅ Correct! R² rose {r2_base:.3f} -> {r2_padded:.3f} just from noise; adjusted R² only {adj_base:.3f} -> {adj_padded:.3f}")

## Exercise 9

**Interpret.** In the model above, `sleep_hours` has a coefficient of about **+2.97**. Which statement is the best interpretation? Set `answer` to a letter.

* **a)** Sleeping one more hour per night *causes* a 3-point rise in exam score.
* **b)** Comparing students with the same study hours, attendance and job status, each extra hour of sleep goes with about 3 more points on average.
* **c)** Students who sleep 1 hour more always score exactly 3 points higher.
* **d)** Sleep explains 3% of the variation in scores.

In [ ]:
answer = "b"

In [ ]:
assert answer == "b", "A coefficient is an average association, 'holding the other included variables fixed'"
print("✅ Correct!")